# 02 Hidden Layer and Loss

Gizli katman (tanh), çıkış katmanı (logits), manuel loss hesaplama ve F.cross_entropy karşılaştırması.


In [ ]:
import torch
import torch.nn.functional as F

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

block_size = 3
X, Y = [], []
for w in words:
    context = [0] * block_size
    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

# Embedding duzlestirme ve gizli katman
emb = C[X]
emb_flat = emb.view(-1, 6)
h = torch.tanh(emb_flat @ W1 + b1)
logits = h @ W2 + b2

# Manuel cross-entropy hesaplama
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
loss_manual = -probs[torch.arange(len(Y)), Y].log().mean()

# F.cross_entropy ile karsilastirma
loss_ce = F.cross_entropy(logits, Y)

print("Manuel loss:", loss_manual.item())
print("F.cross_entropy loss:", loss_ce.item())
print("Fark:", torch.abs(loss_manual - loss_ce).item())
assert torch.allclose(loss_manual, loss_ce), "Loss sonuclari ayni degil!"

# F.cross_entropy neden tercih edilir:
# 1. Numerik kararlilik: Buyuk logitlerde exp() overflow (inf/nan) olusur, cross_entropy max degeri cikarir
extreme_logits = torch.tensor([[50.0, 1000.0, 20.0]])
target = torch.tensor([1])
try:
    bad_counts = extreme_logits.exp()
    bad_probs = bad_counts / bad_counts.sum(1, keepdim=True)
    bad_loss = -bad_probs[0, target].log()
    print("Manuel asiri logit loss:", bad_loss.item())
except Exception as e:
    print("Manuel asiri logit hata:", e)

stable_loss = F.cross_entropy(extreme_logits, target)
print("F.cross_entropy kararlı loss:", stable_loss.item())
# 2. Analitik turev: Fused kernel ara tensorleri bellekte tutmaz, cok daha hizli ve az bellek harcar
